# Apache Iceberg Features: Ingestion Inspection, DML, Time Travel & Evolution in PySpark

This notebook demonstrates key capabilities of Apache Iceberg tables using PySpark and Spark SQL:
1. **Table Ingestion Inspection**: View commit history, snapshots, manifests, and data/metadata files.
2. **DML Demonstration**: Perform direct `INSERT`, `UPDATE`, and `DELETE` operations on the Iceberg table using Spark SQL.
3. **Time Travel**: Query historical table states using specific snapshot IDs and epoch timestamps.
4. **Schema Evolution**: Alter table schemas dynamically (adding, renaming, updating, and dropping columns) without rewriting the table.
5. **Partition Evolution**: Evolve partitioning layout schemas dynamically in-place without losing history or rewriting old data files.

## Step 1: Initialize Spark Session with Apache Iceberg Support

We retrieve the active pre-created Spark session from the Jupyter kernel and configure the GCS Hadoop Catalog (`gcs_hadoop_catalog`) dynamically.

In [ ]:
import os
from pyspark.sql import SparkSession

# REPLACE WITH YOUR GCS STAGING BUCKET NAME
WAREHOUSE_PATH = "gs://YOUR_STAGING_BUCKET/warehouse"

# Get the existing active SparkSession initialized by the Dataproc Jupyter kernel
spark = SparkSession.builder.getOrCreate()

# Set catalog configurations dynamically on the active session
spark.conf.set("spark.sql.catalog.gcs_hadoop_catalog", "org.apache.iceberg.spark.SparkCatalog")
spark.conf.set("spark.sql.catalog.gcs_hadoop_catalog.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog")
spark.conf.set("spark.sql.catalog.gcs_hadoop_catalog.warehouse", WAREHOUSE_PATH)

print("Spark Session configured dynamically with Apache Iceberg Hadoop Catalog support!")

## Step 2: Read and Inspect current Iceberg Data

Let's load the Iceberg table written by our streaming Apache Beam pipeline (`gcs_hadoop_catalog.sensor_db.filtered_readings`) and display the schema and sample records.

In [ ]:
table_name = "gcs_hadoop_catalog.sensor_db.filtered_readings"

# Load table
df = spark.read.table(table_name)
print(f"Total records in {table_name}: {df.count()}")
df.printSchema()
df.show(10, truncate=False)

## Step 3.1: Inspect Table Commit History

Query the `.history` metadata namespace to view the sequence of commits made to the table.

In [ ]:
spark.read.table(f"{table_name}.history").show(truncate=False)

## Step 3.2: Inspect Table Commit Snapshots

Query the `.snapshots` table to view commit metadata, snapshot IDs, and operations (e.g. append, overwrite).

In [ ]:
spark.read.table(f"{table_name}.snapshots").select("committed_at", "snapshot_id", "parent_id", "operation").show(truncate=False)

## Step 3.3: Inspect Table Manifest Files

Query the `.manifests` metadata namespace to view manifest file paths and details.

In [ ]:
spark.read.table(f"{table_name}.manifests").show(5, truncate=False)

## Step 3.4: Inspect Table Data and Metadata Files

Query the `.files` metadata namespace to see the actual Parquet data file paths and record counts.

In [ ]:
spark.read.table(f"{table_name}.files").select("file_path", "file_format", "record_count").show(5, truncate=False)

## Step 4.1: Perform DML - INSERT Record

Insert a new mock reading into the Iceberg table using Spark SQL, and select it to verify the insertion.

In [ ]:
print("Inserting a new record...")
spark.sql(f"INSERT INTO {table_name} VALUES ('device-101', 25.6, 60.1, 'OK', '2026-06-09T22:00:00Z')")

print("Verifying inserted record:")
spark.sql(f"SELECT * FROM {table_name} WHERE device_id = 'device-101'").show()

## Step 4.2: Perform DML - UPDATE Record

Update the humidity value of our inserted record, and verify the modification.

In [ ]:
print("Updating the record's humidity value...")
spark.sql(f"UPDATE {table_name} SET humidity = 62.5 WHERE device_id = 'device-101'")

print("Verifying updated record:")
spark.sql(f"SELECT * FROM {table_name} WHERE device_id = 'device-101'").show()

## Step 4.3: Perform DML - DELETE Record

Delete our inserted record, and verify it no longer exists.

In [ ]:
print("Deleting the inserted record...")
spark.sql(f"DELETE FROM {table_name} WHERE device_id = 'device-101'")

print("Verifying record was deleted:")
spark.sql(f"SELECT * FROM {table_name} WHERE device_id = 'device-101'").show()

## Step 4.4: View Snapshots Created by DML

Inspect the `.snapshots` table to see the new snapshot commits corresponding to our DML queries.

In [ ]:
spark.read.table(f"{table_name}.snapshots").select("committed_at", "snapshot_id", "operation").show(truncate=False)

## Time Travel Setup: Fetch Available Snapshots

Let's fetch the list of snapshots in order to select target snapshot IDs and timestamps for our time travel queries.

In [ ]:
snapshots = spark.read.table(f"{table_name}.snapshots").orderBy("committed_at").collect()
print(f"Found {len(snapshots)} snapshots to query.")

## Step 5.1: Time Travel - Query by First Snapshot ID

Query the table history as of the very first snapshot.

In [ ]:
if len(snapshots) >= 2:
    first_snap = snapshots[0]["snapshot_id"]
    print(f"--- Loading Data from First Snapshot ID: {first_snap} ---")
    spark.read.option("snapshot-id", first_snap).table(table_name).show(5)
else:
    print("Need at least 2 snapshots to demonstrate time travel.")

## Step 5.2: Time Travel - Query by Latest Snapshot ID

Query the table history as of the latest snapshot.

In [ ]:
if len(snapshots) >= 2:
    latest_snap = snapshots[-1]["snapshot_id"]
    print(f"--- Loading Data from Latest Snapshot ID: {latest_snap} ---")
    spark.read.option("snapshot-id", latest_snap).table(table_name).show(5)
else:
    print("Need at least 2 snapshots to demonstrate time travel.")

## Step 5.3: Time Travel - Query by Historical Timestamp

Query the table history as of a specific timestamp (in epoch milliseconds).

In [ ]:
if len(snapshots) >= 2:
    first_snap_ts = int(snapshots[0]["committed_at"].timestamp() * 1000)
    print(f"--- Loading Data as of Timestamp: {first_snap_ts} ms ---")
    spark.read.option("as-of-timestamp", first_snap_ts).table(table_name).show(5)
else:
    print("Need at least 2 snapshots to demonstrate time travel.")

## Step 6.1: Schema Evolution - Add Column

In-place column addition. Adds a new string column `operator_notes` without rewriting existing parquet files.

In [ ]:
print("Adding column 'operator_notes'...")
spark.sql(f"ALTER TABLE {table_name} ADD COLUMN (operator_notes string COMMENT 'Manual notes')")
spark.read.table(table_name).printSchema()

## Step 6.2: Schema Evolution - Rename Column

In-place column renaming. Renames `operator_notes` to `notes`.

In [ ]:
print("Renaming column 'operator_notes' to 'notes'...")
spark.sql(f"ALTER TABLE {table_name} RENAME COLUMN operator_notes TO notes")
spark.read.table(table_name).printSchema()

## Step 6.3: Schema Evolution - Alter Column Comment

Modify the column comment metadata of `notes`.

In [ ]:
print("Updating column 'notes' comment metadata...")
spark.sql(f"ALTER TABLE {table_name} ALTER COLUMN notes COMMENT 'Updated notes comment'")

## Step 6.4: Schema Evolution - Drop Column

Drop the evolved `notes` column from our table.

In [ ]:
print("Dropping column 'notes'...")
spark.sql(f"ALTER TABLE {table_name} DROP COLUMN notes")
spark.read.table(table_name).printSchema()

## Step 7.1: Partition Evolution - Add Partition Field

Evolve our table partitioning layout dynamically by adding the `status` column as a partition field.

In [ ]:
print("Evolving partition spec: adding 'status' field to partitioning layout...")
spark.sql(f"ALTER TABLE {table_name} ADD PARTITION FIELD status")

## Step 7.2: Insert Data and Verify Partition Layout

Let's write new records after adding the partition field, and inspect the `.files` metadata table. We will see that the new Parquet files are organized and tagged under the evolved partition column (`status`), while old files remain untouched (unpartitioned)!

In [ ]:
print("Inserting new records under the evolved partition layout...")
spark.sql(f"INSERT INTO {table_name} VALUES ('device-abc', 28.1, 55.4, 'OK', '2026-06-09T22:10:00Z')")
spark.sql(f"INSERT INTO {table_name} VALUES ('device-xyz', 19.3, 48.2, 'OK', '2026-06-09T22:11:00Z')")

print("Inspecting GCS Parquet files metadata to see partitioning:")
spark.read.table(f"{table_name}.files").select("file_path", "partition", "record_count").show(truncate=False)

## Step 7.3: Partition Evolution - Add Transform Partition Field

Evolve the partition layout further by adding a bucket transform partition field (`bucket(10, device_id)`).

In [ ]:
print("Evolving partition spec: adding 'device_id' bucket(10) transform...")
spark.sql(f"ALTER TABLE {table_name} ADD PARTITION FIELD bucket(10, device_id)")

## Step 7.4: Partition Evolution - Drop Partition Field

Evolve the spec by dropping `status` from partitioning without rewriting any historical files.

In [ ]:
print("Evolving partition spec: dropping 'status' field partition layout...")
spark.sql(f"ALTER TABLE {table_name} DROP PARTITION FIELD status")
print("Partition Layout evolved successfully!")

## Step 8: Shutdown Spark Session

In [ ]:
spark.stop()
print("Spark Session shut down.")